SpaCy & functions

In [1]:
import spacy
nlp = spacy.load("de_core_news_sm")

In [2]:
def root(token):
    if token.dep_ in ["ROOT", "rs", "cd", "ju"]:
        return token
    else:
        return root(token.head)

In [3]:
import copy

def words_to_indices(comment, word_spans):

    index_spans = copy.deepcopy(word_spans)

    spans_id = 0
    span_id = 0

    for i in range(len(comment) + 1):

        # if last characters in sentence are identical
        # doesn't yet deal with repetition of characters 
        # at the end of a span within the comment
        if spans_id >= len(word_spans):
            index_spans[spans_id - 1][1] = i
            continue

        looking_for = word_spans[spans_id][span_id]

        if span_id == 0:
            if comment[i:].startswith(looking_for):
                index_spans[spans_id][span_id] = i
                # move to looking for end of span
                span_id = 1
        else:
            if comment[:i].endswith(looking_for):
                index_spans[spans_id][span_id] = i
                # move to beginning of next span
                spans_id += 1
                span_id = 0

                if spans_id < len(word_spans):
                    looking_for = word_spans[spans_id][span_id]

                    if comment[i:].startswith(looking_for):
                        index_spans[spans_id][span_id] = i
                        # move to looking for end of span
                        span_id = 1

    # dealing with spans that are only one character
    # & thus can't be identified
    for i in range(len(index_spans)):
        if type(index_spans[i][0]) is str:
            index_spans[i][0] = index_spans[i-1][1]
            index_spans[i][1] = index_spans[i][0]
        elif type(index_spans[i][1]) is str:
            index_spans[i][1] = index_spans[i][0]

    return index_spans

In [4]:
def get_spans(comment):

    head = None
    last = None
    spans = []
    current_span = []

    for token in nlp(comment):

        if root(token) != head:

            if last != None: # not needed for first span
                # save end of last span
                current_span.append(last.text)
                spans.append(current_span)

            # start new span
            current_span = [token.text]
            head = root(token)
        
        last = token

    # end span for last token
    current_span.append(last.text)
    spans.append(current_span)
    
    return words_to_indices(comment, spans)


trying out how well span identification works

In [5]:
import pandas as pd

task1 = pd.read_json("Input/Data/train/test_task1.json")
data = task1[task1["flausch"] == "yes"].copy()

In [6]:
dependency_spans = []

for i in range(len(data)):
    dependency_spans.append(get_spans(data.iloc[i]["comment"]))

data["dependency_spans"] = dependency_spans

total_count = 0

for i in range(len(data)):
    sentence_count = 0

    for span in data.iloc[i]["span_pairs"]:
        if span in data.iloc[i]["dependency_spans"]:
            sentence_count += 1
    
    total_count += sentence_count/len(data.iloc[i]["span_pairs"])

print(total_count / len(data))

0.49750812724950644


creating spans for training classifier

In [ ]:
import pandas as pd

path = ("Input/Data/train/")

data_train = pd.read_json(path + "train_task1.json")
data_test = pd.read_json(path + "test_task1.json")

In [ ]:
def add_spans_spacy(data, i):

    phrases = []
    labels = []

    comment = data.iloc[i]["comment"]

    spans = get_spans(comment)

    for j in range(len(spans)):

        span_start = spans[j][0]
        span_end = spans[j][1]

        phrases.append(comment[span_start:span_end])
        labels.append("not")
    
    return [phrases, labels]

In [ ]:
def add_spans_flausch(data, i):

    phrases = []
    labels = []

    comment = data.iloc[i]["comment"]
    types = data.iloc[i]["types"]
    spans = data.iloc[i]["span_pairs"]

    span_end = -1

    for j in range(len(spans)):
        
        if spans[j][0] != span_end + 1:
            phrases.append(comment[span_end + 1:spans[j][0] - 1])
            labels.append("not")

        span_start = spans[j][0]
        span_end = spans[j][1]

        phrases.append(comment[span_start:span_end])
        labels.append(types[j])
    
    if span_end != len(comment):
        phrases.append(comment[span_end + 1:len(comment)])
        labels.append("not")
    
    return [phrases, labels]

In [ ]:
phrases_all = []
labels_all = []

data = data_train

for i in range(len(data)):
    
    if data.iloc[i]["flausch"] == "yes":
        [phrases_i, labels_i] = add_spans_flausch(data, i)     
    else:
        [phrases_i, labels_i] = add_spans_spacy(data, i)

    phrases_all = phrases_all + phrases_i
    labels_all = labels_all + labels_i

training_data = pd.DataFrame(list(zip(phrases_all, labels_all)), columns = ["phrase", "label"])
training_data.to_csv(path + "phrases_with_labels_inlcuding_not_flausch_train.csv", index=False)

In [ ]:
phrases_all = []
labels_all = []

data = data_test

for i in range(len(data)):
    
    if data.iloc[i]["flausch"] == "yes":
        [phrases_i, labels_i] = add_spans_flausch(data, i)     
    else:
        [phrases_i, labels_i] = add_spans_spacy(data, i)

    phrases_all = phrases_all + phrases_i
    labels_all = labels_all + labels_i

test_data = pd.DataFrame(list(zip(phrases_all, labels_all)), columns = ["phrase", "label"])
test_data.to_csv(path + "phrases_with_labels_inlcuding_not_flausch_test.csv", index=False)

preparing data for classification: spaCy spans

In [1]:
import pandas as pd
path = "Input/Data/train/"
data = pd.read_json("test_task1.json")

In [ ]:
# predicted spans

document_column = []
comment_id_column = []
type_column = []
start_column = []
end_column = []
text_column = []

for i in range(len(data)):
    document = data.iloc[i]["document"]
    comment_id = data.iloc[i]["comment_id"]
    comment = data.iloc[i]["comment"]

    spans = get_spans(comment)

    for span in spans:
        document_column.append(document)
        comment_id_column.append(comment_id)
        type_column.append("")
        start_column.append(span[0])
        end_column.append(span[1])
        text_column.append(comment[span[0]:span[1]])

span_data = pd.DataFrame(list(zip(document_column, comment_id_column, type_column, start_column, end_column, text_column)), 
                         columns = ["document", "comment_id", "type", "start", "end", "text"])
span_data.to_csv(path + "span_data.csv", index=False)

classification, saving predictions

In [13]:
#data = span_data
data = pd.read_csv(path + "span_data.csv")

In [ ]:
from transformers import pipeline

pipe = pipeline("text-classification", model="Wiebke/task2_flausch_classification_gbert-large_span_classifier_with_nonspan")

In [ ]:
# update "type" to predicted type for "text"

for i in range(len(data)):
    text = data.iloc[i]["text"]

    # update type (label)
    data.loc[i, "type"] = pipe(text)[0]["label"]

In [ ]:
import csv

submission_path = "predictions/"

# get prediction data into right format for submission
# keep only flausch spans & remove text column
pred_data = data[data["type"] != "not"].drop(columns=["text"])

pred_data.to_csv(submission_path + "task2-predicted-spacy.csv", index=False, quoting=csv.QUOTE_ALL)

Evaluation

In [55]:
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support
from multiset import *

# Labels for the fine grained classification
ALL_LABELS = ["affection declaration","agreement","ambiguous",
              "compliment","encouragement","gratitude","group membership",
              "implicit","positive feedback","sympathy"]

# TASK 1
def binary_eval(file_gold, file_pred):

    test_g = pd.read_csv(file_gold)
    test_p = pd.read_csv(file_pred)

    (bin_prec, bin_rec, bin_f, bin_supp) = precision_recall_fscore_support(test_g.flausch, test_p.flausch, pos_label="yes", average='binary')

    print("TASK 1: BINARY CLASSIFICATION",
          "\n=============================",
          "\nPrecision:\t %.4f" % bin_prec,
          "\nRecall:\t\t %.4f" % bin_rec,
          "\nF-score:\t %.4f" % bin_f)

    return((bin_prec, bin_rec, bin_f))


def fine_grained_flausch_by_label(file_gold, file_pred):

    # read files
    gold = pd.read_csv(file_gold)
    gold['cid']= gold['document']+"_"+gold['comment_id'].apply(str)
    predicted = pd.read_csv(file_pred)
    predicted['cid']= predicted['document']+"_"+predicted['comment_id'].apply(str)


    # annotation sets (predicted)
    pred_spans = Multiset()
    pred_spans_loose = Multiset()
    pred_types = Multiset()

    # annotation sets (gold)
    gold_spans = Multiset()
    gold_spans_loose = Multiset()
    gold_types = Multiset()

    for row in predicted.itertuples(index=False):
        pred_spans.add((row.cid,row.type,row.start,row.end))
        pred_spans_loose.add((row.cid,row.start,row.end))
        pred_types.add((row.cid,row.type))
    for row in gold.itertuples(index=False):
        gold_spans.add((row.cid,row.type,row.start,row.end))
        gold_spans_loose.add((row.cid,row.start,row.end))
        gold_types.add((row.cid,row.type))

    # precision = true_pos / true_pos + false_pos
    # recall = true_pos / true_pos + false_neg
    # f_1 = 2 * prec * rec / (prec + rec)

    results = {'TOTAL': {'STRICT': {},'SPANS': {},'TYPES': {}}}
    # label-wise evaluation (only for strict and type)
    for label in ALL_LABELS:
        results[label] = {'STRICT': {},'TYPES': {}}
        gold_spans_x = set(filter(lambda x: x[1].__eq__(label), gold_spans))
        pred_spans_x = set(filter(lambda x: x[1].__eq__(label), pred_spans))
        gold_types_x = set(filter(lambda x: x[1].__eq__(label), gold_types))
        pred_types_x = set(filter(lambda x: x[1].__eq__(label), pred_types))

        # strict: spans + type must match
        ### NOTE: x and y / x returns 0 if x = 0 and y/x otherwise (test for zero division)
        strict_p = float(len(pred_spans_x)) and float( len(gold_spans_x.intersection(pred_spans_x))) / len(pred_spans_x)
        strict_r = float(len(gold_spans_x)) and float( len(gold_spans_x.intersection(pred_spans_x))) / len(gold_spans_x)
        strict_f = (strict_p + strict_r) and 2 * strict_p * strict_r / (strict_p + strict_r)
        results[label]['STRICT']['prec'] = strict_p
        results[label]['STRICT']['rec'] = strict_r
        results[label]['STRICT']['f1'] = strict_f

        # detection mode: only types must match (per post)
        types_p = float(len(pred_types_x)) and float( len(gold_types_x.intersection(pred_types_x))) / len(pred_types_x)
        types_r = float(len(gold_types_x)) and float( len(gold_types_x.intersection(pred_types_x))) / len(gold_types_x)
        types_f = (types_p + types_r) and 2 * types_p * types_r / (types_p + types_r)
        results[label]['TYPES']['prec'] = types_p
        results[label]['TYPES']['rec'] = types_r
        results[label]['TYPES']['f1'] = types_f

    # Overall evaluation
    # strict: spans + type must match
    strict_p = float(len(pred_spans)) and float( len(gold_spans.intersection(pred_spans))) / len(pred_spans)
    strict_r = float(len(gold_spans)) and float( len(gold_spans.intersection(pred_spans))) / len(gold_spans)
    strict_f = (strict_p + strict_r) and 2 * strict_p * strict_r / (strict_p + strict_r)
    results['TOTAL']['STRICT']['prec'] = strict_p
    results['TOTAL']['STRICT']['rec'] = strict_r
    results['TOTAL']['STRICT']['f1'] = strict_f

    # spans: spans must match
    spans_p = float(len(pred_spans_loose)) and float( len(gold_spans_loose.intersection(pred_spans_loose))) / len(pred_spans_loose)
    spans_r = float(len(gold_spans_loose)) and float( len(gold_spans_loose.intersection(pred_spans_loose))) / len(gold_spans_loose)
    spans_f = (spans_p + spans_r) and 2 * spans_p * spans_r / (spans_p + spans_r)
    results['TOTAL']['SPANS']['prec'] = spans_p
    results['TOTAL']['SPANS']['rec'] = spans_r
    results['TOTAL']['SPANS']['f1'] = spans_f

    # detection mode: only types must match (per post)
    types_p = float(len(pred_types)) and float( len(gold_types.intersection(pred_types))) / len(pred_types)
    types_r = float(len(gold_types)) and float( len(gold_types.intersection(pred_types))) / len(gold_types)
    types_f = (types_p + types_r) and 2 * types_p * types_r / (types_p + types_r)
    results['TOTAL']['TYPES']['prec'] = types_p
    results['TOTAL']['TYPES']['rec'] = types_r
    results['TOTAL']['TYPES']['f1'] = types_f

    print("STRICT:\n ",strict_p,strict_r,strict_f)
    print("SPANS:\n ",spans_p,spans_r,spans_f)
    print("TYPES:\n ",types_p,types_r,types_f)
    return(results)

In [ ]:
path = "Input/Data/train/"
submission_path = "predictions/"

file_gold = path + "gold_data_task2.csv"
file_pred = submission_path + "task2-predicted" + ".csv"
results = fine_grained_flausch_by_label(file_gold, file_pred)

gold_data_task2 = pd.read_csv(file_gold)


typeList = gold_data_task2["type"].unique()
all = []
for t in typeList:
    print(t)
    print(pd.DataFrame(results[t]))
    print()

STRICT:
  0.39578630549285176 0.34673698088332233 0.3696416022487702
SPANS:
  0.4161023325808879 0.36453526697429134 0.3886156008432888
TYPES:
  0.7848006019563581 0.6875411997363217 0.7329585382993675
positive feedback
        STRICT     TYPES
prec  0.443787  0.825210
rec   0.369004  0.740573
f1    0.402955  0.780604

affection declaration
        STRICT     TYPES
prec  0.479070  0.822115
rec   0.408730  0.750000
f1    0.441113  0.784404

group membership
        STRICT     TYPES
prec  0.214286  0.692308
rec   0.142857  0.500000
f1    0.171429  0.580645

encouragement
        STRICT     TYPES
prec  0.235955  0.869048
rec   0.241379  0.879518
f1    0.238636  0.874251

compliment
        STRICT     TYPES
prec  0.314815  0.790123
rec   0.318352  0.810127
f1    0.316574  0.800000

ambiguous
        STRICT     TYPES
prec  0.400000  0.400000
rec   0.272727  0.272727
f1    0.324324  0.324324

implicit
        STRICT     TYPES
prec  0.250000  0.500000
rec   0.090909  0.181818
f1    0.133333  